<a href="https://colab.research.google.com/github/Panfordd/lab-4-llm-decision-support/blob/main/Lab%204_llm_support_decision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


# Section 1 — Talking to an LLM Programmatically

**Part 1.1 — Your first API call**

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
          temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user",   "content": user_prompt},
      ],
      temperature=temperature,
      max_tokens=max_tokens,
  )
  return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
response=ask_llm("What is a balance?")
print(response)

# TODO: Print response.usage as well — how many tokens did your call consume?
response = client.chat.completions.create(
        model=MODEL,
       messages=[
           {"role": "system", "content": "You are a helpful assistant"},
           {"role": "user",   "content": "What is a balance?"},
       ],
       temperature=0.7,
       max_tokens=500,
   )
print(response.usage)

A balance can refer to different concepts depending on the context. Here are a few possible meanings:

1. **Physical balance**: In physics, balance refers to the state of equilibrium, where the weight or force of an object is evenly distributed, allowing it to remain stable and upright. For example, a seesaw is in balance when the weights on both sides are equal.
2. **Financial balance**: In finance, a balance refers to the amount of money in an account, such as a bank account or credit card account. It can also refer to the state of having equal income and expenses, or a stable financial situation.
3. **Work-life balance**: This refers to the balance between an individual's work and personal life, where they have enough time and energy for both their professional and personal responsibilities.
4. **Emotional balance**: In psychology, emotional balance refers to a state of mental well-being, where an individual is able to manage their emotions, cope with stress, and maintain a positive

**Student Reasoning — Anatomy of a call** 1. What is the difference between the system and user roles? Give an example of something that belongs in each. 2. What is a token, roughly? Why do API providers bill per token rather than per request?

**Part 1.2 — Temperature: the randomness dial**

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
for temperature in [0.0, 1.2]:
  print("\nTemperature: " + str(temperature))
  for i in range(5):
    response = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=temperature)
    print("Response " + str(i + 1) + ": " + response)
    print()

# TODO: Print all 10 answers, grouped by temperature.


Temperature: 0.0
Response 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "save" or "keep", so this name is simple and straightforward.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes th

**Student Reasoning — Temperature **What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

# Section 2 — The Dataset: Loan Application Letters

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


# Section 3 — Prompt Engineering for the Decision Support System

# Part 3.1 — Component 1: Summarization

In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1= "Summarize this:"

summaries_v1 = {}
for letter_id in ["L002", "L006"]:
  letter_text = LETTERS[letter_id]

  user_prompt = SUMMARY_PROMPT_V1 + "\n\n" + letter_text

  answer = ask_llm(user_prompt)
  summaries_v1[letter_id] = answer


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_PROMPT_V2 = """You are an assistant to a microfinance loan officer.
                       Summarise the applicant's loan request in a factual and neutral way

                       Requirements:
                       1. Use only information stated in the loan application
                       2. Do not invent or assume any details
                       3. Keep the summary between 3-4 sentences
                       Include important facts about the applicant, loan amount, purpose,repayment information, and financial situation where available"""

summaries_v2 = {}
for letter_id in["L002", "L006"]:
  letter_text = LETTERS[letter_id]

  user_prompt = SUMMARY_PROMPT_V2 + "\n\n"+ letter_text

  answer = ask_llm(user_prompt,
                 system_prompt = SUMMARY_PROMPT_V2,
                 temperature =0)
  summaries_v2[letter_id] = answer


# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n Comparing V1 and V2 Summaries ")
for letter_id in ["L002", "L006"]:
    print("\n" + letter_id )
    print("V1 SUMMARY: \n" + summaries_v1[letter_id])
    print("V2 SUMMARY:\n" + summaries_v2[letter_id])


 Comparing V1 and V2 Summaries 

L002
V1 SUMMARY: 
Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and promises to repay the loan as soon as possible, despite not having collateral.
V2 SUMMARY:
Kwame Boateng, a commercial driver from Kumasi, has applied for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season, but has not specified a repayment schedule, stating only that he can pay back whenever the money comes. At present, Mr. Boateng does not have any collateral to secure the loan.

L006
V1 SUMMARY: 
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience in these ventures, but c

**Student Reasoning — Summarization **prompts 1. What concrete problems did V1's output have that V2 fixed? Quote examples. 2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

**Part 3.2 — Component 2: Structured extraction (JSON)**

In [11]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

import json
import pandas as pd

EXACT_PROMPT = """You are an information extraction assistant for a microfinance loan officer
                  Extract information from the loan application and return only a valid JSON object.

                  Use these keys:

                  {
                  "applicant_name":string,
                  "amount_ghs":number,
                  "purpose":string,
                  "monthly_profit_ghs":number or null,
                  "has_collateral_or_guarantor":boolean,
                  "repayment_months":number or null
                  }

                  Rules:
                  1. Use only information explicitly stated in the application.
                  2. Do not invent inormation.
                  3. If a field is not stated use null.
                  4. Return numbers without GHS symbols or commas.
                  5. has_collateral_or_guarantor must be true or false.
                  6. Return only the JSON object. Do not add explantions.

                  Example:

                  Loan application:
                  My name is Olivia Panford. I am a student at Ashesi University and I am requesting GHS 10,000
                  to pay my tuition fees. I lost my dad recently but I will gather funds from my business  to repay the loan
                  over 12 months. My mother will act as my guarantor.

                  Output:

                  {"applicant_name":"Olivia Panford",
                  "amount_ghs":10000,
                  "purpose":"pay tuition fees",
                  "monthly_profit_ghs":null,
                  "has_collateral_or_guarantor":true,
                  "repayment_months":12}    """


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text):

    user_prompt = "Extract the required fields from this loan application:\n\n" + letter_text

    answer = ask_llm(
        user_prompt,
        system_prompt=EXACT_PROMPT,
        temperature=0
    )

    answer = answer.strip()

    if answer.startswith("```json"):
        answer = answer[7:]

    if answer.startswith("```"):
        answer = answer[3:]

    if answer.endswith("```"):
        answer = answer[:-3]

    answer = answer.strip()

    try:
        extracted_info = json.loads(answer)
        return extracted_info

    except json.JSONDecodeError:
        print("Warning: Invalid JSON returned")
        return None
# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

extracted_results = []

for letter_id, letter_text in LETTERS.items():
    data = extract_fields(letter_text)

    if data is not None:
        data["Letter_id"] = letter_id
        extracted_results.append(data)


extracted_df = pd.DataFrame(extracted_results)

extracted_df







,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,Letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,buy feed and 500 new layers for poultry farm,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


# Student Reasoning — Structured extraction
1. Why must the few-shot example NOT come from the six letters you are processing? 2. Why "use null, do not guess" — what did the model do without that instruction? 3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?

**Part 3.3 — Component 3: The decision-support brief**

In [13]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Using the original loan application and the extracted information,
prepare a decision-support brief.

Your response must contain:

1. Strengths
- Give bullet points.
- Use only strengths supported by the application.

2. Risks / Red Flags
- Give bullet points.
- Identify concerns supported by the application.
- Do not invent information.

3. Missing Information
- State important information that is missing and that the loan officer
  should request from the applicant.

4. Suggested Next Step
- Suggest an appropriate next action such as:
  invite for interview, request supporting documents, or flag for senior review.
- Do NOT say approve or reject.

The purpose of this system is to support the loan officer, not make the final decision.
Final lending decisions must always be made by a human.
"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
def generate_brief(letter_text, extracted_data):

    user_prompt = (
        "Original loan application:\n\n"
        + letter_text
        + "\n\nExtracted information:\n\n"
        + json.dumps(extracted_data, indent=2)
    )

    answer = ask_llm(
        user_prompt,
        system_prompt=BRIEF_PROMPT,
        temperature=0
    )

    return answer


brief_results = {}

for data in extracted_results:

    letter_id = data["Letter_id"]
    letter_text = LETTERS[letter_id]

    brief = generate_brief(letter_text, data)

    brief_results[letter_id] = brief


for letter_id in ["L001", "L002", "L006"]:

    print("=" * 70)
    print("DECISION-SUPPORT BRIEF FOR " + letter_id)
    print("=" * 70)

    print(brief_results[letter_id])
    print()


DECISION-SUPPORT BRIEF FOR L001
Decision-Support Brief:

**Strengths:**
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
* The applicant has a proven track record of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution.
* The applicant has a guarantor, her sister, who is a teacher, providing an added layer of security for the loan.
* The applicant has a clear plan for repayment, proposing to pay GHS 450 monthly over 20 months.

**Risks / Red Flags:**
* The applicant's proposed monthly repayment of GHS 450 is approximately half of her current monthly profit, which may leave limited room for unexpected expenses or business downturns.
* The loan amount of GHS 8,000 is significant compared to the applicant's current monthly profit and savings, which may pose a risk if the business expansion does not generate sufficient additional income.

**Missing Information:*

**Part 3.4-Commit your prompt templates**



Commit hash:
537c328

# Section 4 — Evaluation: Quality, Reliability, Appropriateness

**Part 4.1 — Extraction accuracy against gold**

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
